# S6.4 — Document Grading Node

Binary relevance grading node for the agentic RAG workflow.
Evaluates each retrieved document against the user's query using structured LLM output.

In [1]:
import sys
sys.path.insert(0, "../..")

## 1. Imports & Module Verification

In [2]:
from src.services.agents.nodes.grade_documents_node import (
    GRADING_PROMPT,
    ainvoke_grade_documents_step,
    continue_after_grading,
)
from src.services.agents.models import GradeDocuments, GradingResult, SourceItem
from src.services.agents.state import AgentState
from src.services.agents.context import AgentContext

assert callable(ainvoke_grade_documents_step)
assert callable(continue_after_grading)
assert isinstance(GRADING_PROMPT, str)
print("\u2713 All imports successful")

✓ All imports successful


## 2. GRADING_PROMPT Template

In [3]:
print(GRADING_PROMPT)
print("---")
formatted = GRADING_PROMPT.format(
    query="How do transformers work?",
    document="Transformers use self-attention mechanisms to process sequences in parallel."
)
assert "{query}" not in formatted
assert "{document}" not in formatted
assert "How do transformers work?" in formatted
print("\u2713 Prompt template formats correctly")

You are a document relevance grader for an academic research assistant.

Given a user query and a retrieved document chunk, assess whether the document is relevant
to answering the query.

Grade as "yes" if the document contains information that would help answer the query.
Grade as "no" if the document is not relevant or does not contain useful information.

User query: {query}

Document content:
{document}

Provide your binary grade ("yes" or "no") and a brief reasoning.
---
✓ Prompt template formats correctly


## 3. Grade Documents with Mocked LLM

In [4]:
from unittest.mock import AsyncMock, MagicMock
from langchain_core.messages import HumanMessage

# Create mock LLM that returns structured grades
provider = MagicMock()
mock_model = MagicMock()
mock_structured = AsyncMock()
mock_structured.ainvoke = AsyncMock(side_effect=[
    GradeDocuments(binary_score="yes", reasoning="Directly about transformers"),
    GradeDocuments(binary_score="no", reasoning="Not about transformers"),
    GradeDocuments(binary_score="yes", reasoning="Related to attention mechanisms"),
])
mock_model.with_structured_output.return_value = mock_structured
provider.get_langchain_model.return_value = mock_model

ctx = AgentContext(llm_provider=provider, model_name="test-model")

sources = [
    SourceItem(arxiv_id="1706.03762", title="Attention Is All You Need",
               authors=["Vaswani"], url="https://arxiv.org/abs/1706.03762",
               chunk_text="Transformers use self-attention."),
    SourceItem(arxiv_id="9999.99999", title="Unrelated Paper",
               authors=["Smith"], url="https://arxiv.org/abs/9999.99999",
               chunk_text="This paper is about cooking."),
    SourceItem(arxiv_id="1810.04805", title="BERT",
               authors=["Devlin"], url="https://arxiv.org/abs/1810.04805",
               chunk_text="BERT uses bidirectional attention."),
]

state = AgentState(
    messages=[HumanMessage(content="How do transformers work?")],
    original_query="How do transformers work?",
    sources=sources,
    retrieval_attempts=1,
    grading_results=[],
    relevant_sources=[],
)

result = await ainvoke_grade_documents_step(state, ctx)

print(f"Grading results: {len(result['grading_results'])}")
for gr in result["grading_results"]:
    print(f"  {gr.document_id}: relevant={gr.is_relevant}, score={gr.score}, reason={gr.reasoning}")

print(f"\nRelevant sources: {len(result['relevant_sources'])}")
for src in result["relevant_sources"]:
    print(f"  {src.arxiv_id}: {src.title}")

assert len(result["relevant_sources"]) == 2
assert result["relevant_sources"][0].arxiv_id == "1706.03762"
assert result["relevant_sources"][1].arxiv_id == "1810.04805"
print("\n✓ Mixed grading works correctly")

Grading results: 3
  1706.03762: relevant=True, score=1.0, reason=Directly about transformers
  9999.99999: relevant=False, score=0.0, reason=Not about transformers
  1810.04805: relevant=True, score=1.0, reason=Related to attention mechanisms

Relevant sources: 2
  1706.03762: Attention Is All You Need
  1810.04805: BERT

✓ Mixed grading works correctly


## 4. Conditional Edge Routing

In [5]:
# Case 1: Has relevant sources → generate
state1 = AgentState(relevant_sources=[sources[0]], retrieval_attempts=1)
ctx1 = AgentContext(llm_provider=MagicMock(), max_retrieval_attempts=3)
assert continue_after_grading(state1, ctx1) == "generate"
print("\u2713 Has relevant sources → generate")

# Case 2: No relevant sources, retries left → rewrite
state2 = AgentState(relevant_sources=[], retrieval_attempts=1)
assert continue_after_grading(state2, ctx1) == "rewrite"
print("\u2713 No relevant, retries left → rewrite")

# Case 3: No relevant sources, retries exhausted → generate (force)
state3 = AgentState(relevant_sources=[], retrieval_attempts=3)
assert continue_after_grading(state3, ctx1) == "generate"
print("\u2713 Retries exhausted → generate (forced)")

Routing to generate: retries exhausted (3/3)


✓ Has relevant sources → generate
✓ No relevant, retries left → rewrite
✓ Retries exhausted → generate (forced)


## 5. Empty Sources Edge Case

In [6]:
provider2 = MagicMock()
ctx2 = AgentContext(llm_provider=provider2, model_name="test-model")
empty_state = AgentState(
    messages=[HumanMessage(content="test")],
    sources=[],
)

result2 = await ainvoke_grade_documents_step(empty_state, ctx2)

assert result2["grading_results"] == []
assert result2["relevant_sources"] == []
provider2.get_langchain_model.assert_not_called()
print("✓ Empty sources → no LLM calls, empty results")

✓ Empty sources → no LLM calls, empty results


## Summary

S6.4 Document Grading Node provides:
- `ainvoke_grade_documents_step` — grades each source via LLM structured output
- `continue_after_grading` — routes to generate or rewrite based on results
- Graceful LLM failure handling (marks failed docs as not relevant)
- Empty sources short-circuit (no LLM calls)
- Exhausted retries force generation even without relevant docs